In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import h5py
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader,random_split
from tqdm import trange, tqdm
from sklearn.metrics import precision_score
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
import math
from torchvision.transforms import transforms as T
from torch.nn import CrossEntropyLoss

In [ ]:
path = "/content/drive/MyDrive/CERN/Dataset_Specific_labelled.h5"

In [ ]:
class JetDataset(Dataset):
    def __init__(self, path, augment=False):
        self.file = h5py.File(path, "r")
        self.images = self.file["jet"]
        self.labels = self.file["Y"]
        self.augment = augment
        self.aug_transforms = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=15)
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        image = image.permute(2, 0, 1)
        if self.augment:
            image = self.aug_transforms(image)
        image = (image - image.mean()) / (image.std() + 1e-6)
        return image, label

In [ ]:
total_size = len(JetDataset(path))
train_size = int(0.7 * total_size)
val_size   = int(0.15 * total_size)
test_size  = total_size - train_size - val_size

base_dataset = JetDataset(path, augment=False)

train_idx, val_idx, test_idx = random_split(
    range(total_size),
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
from torch.utils.data import Subset
train_dataset = Subset(JetDataset(path, augment=True), train_idx.indices)
val_dataset   = Subset(JetDataset(path, augment=False), val_idx.indices)
test_dataset  = Subset(JetDataset(path, augment=False), test_idx.indices)

In [ ]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

In [ ]:
class patchEmbedding(nn.Module):
  def __init__(self,patch_size,in_channels,embed_dim):
    super().__init__()
    self.proj = nn.Conv2d(
        in_channels = in_channels,
        out_channels = embed_dim,
        stride = patch_size,
        kernel_size = patch_size
    )
  def forward(self,x):

    ## assuming the shape of x = (batch size,channels,height,width)
    x = self.proj(x) # (batch_size,768,14,14)
    #print("proj",x.shape)
    x = x.flatten(2,-1) # (batch,768,196)
    #print("flatten",x.shape)
    x = x.transpose(1,2) # (batch_size,196,768)
    #print("transpose",x.shape)
    return x

In [ ]:
patch = patchEmbedding(5,8,200)

In [ ]:
patch(images).shape

In [ ]:
class PositionalEncoding(nn.Module):
  def __init__(self,patches,embed_dim):
    super().__init__()
    ## img = (3,224,224) => (batch_size,patches,embed_dim)
    self.d_k = embed_dim
    self.patches = patches + 1
    pe = torch.zeros(self.patches,self.d_k) # (197,768)
    div = torch.exp(torch.arange(0,self.d_k,2).float() * -(math.log(1000.0)/self.d_k)) # (1,384)
    ### range(0,df-2,2) => 2i
    ### -(math.log(1000.0)/self.d_k) => -log(1000.0)/d_k
    ### multiplication => (2i/d_k) * -log(100.000)
    ### exp = -1000.0^(2i/d_k) => 1/1000.o^(2i/dk)
    position = torch.arange(0,self.patches).float().unsqueeze(1) # (197,1)
    #print("before position")
    pe[:,0::2] = torch.sin(position*div) # only even positions are taken (197,1)*(1,384) = (197,384)
    pe[:,1::2] = torch.cos(position*div) # only the odd positions are taken(197,384)
    # (batch,197,384)
    #print("pe done ")
    pe = pe.unsqueeze(0) # adding the batch parameter
    self.register_buffer("pe",pe) # setting in the buffer to avoid the backtracking
  def forward(self,x):
    return x + (self.pe[:,:x.size(1),:]).requires_grad_(False) # adding the positional vector => :self.patches are added to avoid the upscale in that dim

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self,seq_len,embedding_dim,heads,head_dim,num_chunks,chunk_size):
    super().__init__()
    #assert seq_len == math.floor(seq_len / chunk_size) *chunk_size
    assert embedding_dim == heads * head_dim
    self.heads = heads
    self.head_dim = head_dim
    self.chunk_size = chunk_size
    self.num_chunks = math.ceil(seq_len / chunk_size)
    self.Q = nn.Linear(embedding_dim,embedding_dim)
    self.K = nn.Linear(embedding_dim,embedding_dim)
    self.V = nn.Linear(embedding_dim,embedding_dim)

  def forward(self,x):
    # x = (32,100,128)
    B,L,E =  x.shape
    #print(type(L))
    q = (self.Q(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2) # (32,100,128) => (32,100,2,64) => (32,2,100,64)
    k = (self.K(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2)
    v = (self.V(x)).view(B,L, self.heads, self.head_dim).transpose(1, 2)
    d_k = q.shape[-1]
    attention_scores = F.softmax((q @ k.transpose(-2,-1)) / math.sqrt(d_k),dim=-1)
    attention = attention_scores @ v
    attention = attention.transpose(1,2).contiguous().view(B,L,self.heads*self.head_dim) # (32,5,156,50)
    return attention

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self):
    super().__init__()
    self.eps = 0.01
    self.alpha = nn.Parameter(torch.ones(1))
    self.beta = nn.Parameter(torch.ones(1))
    pass
  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    std = x.std(dim=-1,keepdim=True)
    return (self.alpha * ((x - mean)/(std+self.eps))) + self.beta

In [ ]:
class FeedForwardlayer(nn.Module):
  def __init__(self,patches,embed_dim):
    super().__init__()
    self.dim = 348
    self.fc_1 = nn.Linear(embed_dim,self.dim)
    self.fc_2 = nn.Linear(self.dim,embed_dim)
    self.relu = nn.ReLU()
  def forward(self,x):
    return self.fc_2(self.relu(self.fc_1(x)))

In [ ]:
class ResidualBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm = LayerNorm()
  def forward(self,X,sublayer):
    #print("in residual ")
    return X + sublayer(self.layer_norm(X)) # ada and norm
    ## here the sublayer is normed first then add

In [ ]:
class EncoderBlock(nn.Module):
  def __init__(self,attention:MultiHeadAttention,fnn:FeedForwardlayer,patches,embed_dim):
    super().__init__()
    self.attention = attention
    self.fnn = fnn
    self.patches = patches
    self.embed_dim = embed_dim
    self.residual = nn.ModuleList([ResidualBlock() for _ in range(2)]) # we want and 2 residual layer connected
  def forward(self,X):
    X = self.residual[0](X,lambda x:self.attention(X)) ## here the attention is second layer (sublayer) and the X is the input which is going to add
    X = self.residual[1](X,lambda x:self.fnn(X))
    return X

In [ ]:
class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNorm()

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        #print("encoder Completed")
        return self.norm(x)

In [ ]:
class ViTClassifier(nn.Module):
    def __init__(self,embeded_dim,num_classes):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(embeded_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(2048, num_classes)
        )

    def forward(self, x):
        return self.head(x)

In [ ]:
class ViT(nn.Module):
    def __init__(self, patches=625,patch_size=5, embed_dim=200, num_classes=2):
        super().__init__()
        self.patch_embedding = patchEmbedding(patch_size, 8, embed_dim)
        self.positional = PositionalEncoding(patches, embed_dim)
        self.encoder = encoder
        self.classifier = classifier
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x):
        x = self.patch_embedding(x)
        #print(x.shape)
        #print("cls",cls_token.shape)
        #print(x.shape)
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), x], dim=1)
        #print(x.shape)
        x = self.positional(x)
        x = self.encoder(x)
        cls_features = x[:, 0, :]  # Take CLS token
        return self.classifier(cls_features)


In [ ]:
patches = 625
embed_dim =  200
num_classes = 2
patch_size = 5
heads = 5
in_channels = 8
head_dim = 40
num_chunks = 5
chunk_size = 5
patch_embedding = patchEmbedding(patch_size,in_channels,embed_dim)
positional = PositionalEncoding(patches,embed_dim)
attention = MultiHeadAttention(patches,embed_dim,heads,head_dim,num_chunks,chunk_size)
residual = ResidualBlock()
fnn = FeedForwardlayer(patches,embed_dim)
encoderBlock = EncoderBlock(attention,fnn,patches,embed_dim)
encoder = Encoder(nn.ModuleList(12*[encoderBlock]))
classifier = ViTClassifier(embed_dim,num_classes)

In [ ]:
model = ViT(patches,patch_size,embed_dim,num_classes)

In [ ]:
model

In [ ]:
torch.cuda.empty_cache()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ViT(patches, patch_size, embed_dim, num_classes=2).to(device)  # 2 classes!
epochs = 15
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)  # Better optimizer
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",patience=2)

# Storage lists
train_losses, val_losses = [], []
train_accs, val_accs = [], []
train_precs, val_precs = [], []
best_val_acc = 0
for epoch in trange(epochs, desc="Training Progress"):
    model.train()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    for batch_idx, (image, label) in enumerate(tqdm(train_loader, desc=f"Train {epoch}", leave=False)):
        image, label = image.to(device), label.to(device)
        label = label.squeeze(1).long()
        optimizer.zero_grad()
        #print(image.shape,label.shape)
        pred = model(image)  # [batch, 2]
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Accuracy
        _, predicted = torch.max(pred, 1)
        total += label.size(0)
        correct += (predicted == label).sum().item()

        # Precision collection
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(label.cpu().numpy())

    # Epoch metrics
    train_loss = total_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_prec = precision_score(all_labels, all_preds, average='binary', zero_division=0)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_precs.append(train_prec)
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for image, label in tqdm(val_loader, desc=f"Val {epoch}", leave=False):
            image, label = image.to(device), label.to(device)
            label = label.squeeze(1).long()
            pred = model(image)
            loss = criterion(pred, label)
            val_loss += loss.item()

            _, predicted = torch.max(pred, 1)
            val_total += label.size(0)
            val_correct += (predicted == label).sum().item()

            val_preds.extend(predicted.cpu().numpy())
            val_labels.extend(label.cpu().numpy())

    val_loss_avg = val_loss / len(val_loader)
    val_acc = 100. * val_correct / val_total
    val_prec = precision_score(val_labels, val_preds, average='binary', zero_division=0)

    val_losses.append(val_loss_avg)
    val_accs.append(val_acc)
    val_precs.append(val_prec)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict":scheduler.state_dict(),
            "loss": loss
        }, "checkpoint.pth")

    scheduler.step(val_loss)
    print(f"Epoch {epoch+1:02d}/{epochs}")
    print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%, Prec={train_prec:.3f}")
    print(f"  Val:   Loss={val_loss_avg:.4f}, Acc={val_acc:.2f}%, Prec={val_prec:.3f} | Best: {best_val_acc:.2f}%")
    print(f"  LR: {scheduler.get_last_lr()[0]:.2e}")
    print("-" * 60)


In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12))

ax1.plot(train_losses, label='Train Loss', linewidth=2)
ax1.plot(val_losses, label='Val Loss', linewidth=2)
ax1.set_title('Loss')
ax1.legend()

ax2.plot(train_accs, label='Train Acc', linewidth=2)
ax2.plot(val_accs, label='Val Acc', linewidth=2)
ax2.set_title('Accuracy (%)')
ax2.legend()

ax3.plot(train_precs, label='Train Prec', linewidth=2)
ax3.plot(val_precs, label='Val Prec', linewidth=2)
ax3.set_title('Precision')
ax3.legend()

plt.tight_layout()
plt.savefig('vit_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n🎉 Final Best Val Acc: {best_val_acc:.2f}%")
print(f"✅ Best model saved: best_vit_jet_model.pth")
